# Library

In [8]:
import pandas as pd
import numpy as np
import os

# 데이터가 있는 폴더 경로 (본인 환경에 맞게 수정)
INPUT_DIR = '../provided'
OUTPUT_DIR = './' # preprocessed 폴더 위치


# Base Data Preprocessing

## base_seeds_features

In [9]:
# 원본 시드 데이터 로드
m_seeds = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneySeeds.csv'))
w_seeds = pd.read_csv(os.path.join(INPUT_DIR, 'WNCAATourneySeeds.csv'))

# 문자열에서 숫자 부분만 추출하는 함수
def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)
w_seeds['SeedNum'] = w_seeds['Seed'].apply(parse_seed)

# 필요한 컬럼만 추출하여 저장
base_seeds_m = m_seeds[['Season', 'TeamID', 'SeedNum']]
base_seeds_w = w_seeds[['Season', 'TeamID', 'SeedNum']]

base_seeds_m.to_csv(os.path.join(OUTPUT_DIR, 'base_seeds_features_M.csv'), index=False)
base_seeds_w.to_csv(os.path.join(OUTPUT_DIR, 'base_seeds_features_W.csv'), index=False)
print("A 파트 저장 완료: base_seeds_features_M.csv / base_seeds_features_W.csv")


A 파트 저장 완료: base_seeds_features_M.csv / base_seeds_features_W.csv


## base_season_stats_features

In [10]:
# 원본 정규시즌 성적 로드
m_reg = pd.read_csv(os.path.join(INPUT_DIR, 'MRegularSeasonCompactResults.csv'))
w_reg = pd.read_csv(os.path.join(INPUT_DIR, 'WRegularSeasonCompactResults.csv'))

def build_season_stats(reg_df):
    # 이긴 팀 관점 데이터 추가
    w = reg_df[['Season', 'WTeamID', 'WScore', 'LScore']].copy()
    w.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    w['Win'] = 1
    
    # 진 팀 관점 데이터 추가
    l = reg_df[['Season', 'LTeamID', 'LScore', 'WScore']].copy()
    l.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    l['Win'] = 0
    
    # 병합
    df = pd.concat([w, l], ignore_index=True)
    df['Margin'] = df['ScoreFor'] - df['ScoreAgainst']
    
    # 그룹화 및 통계 수치(Features) 집계
    stats = df.groupby(['Season', 'TeamID']).agg(
        Games=('Win', 'count'),
        Wins=('Win', 'sum'),
        AvgScore=('ScoreFor', 'mean'),
        AvgMargin=('Margin', 'mean'),
    ).reset_index()
    
    stats['WinRate'] = stats['Wins'] / stats['Games']
    
    # 문서 명세에 맞춰 컬럼 순서 정리
    return stats[['Season', 'TeamID', 'Games', 'Wins', 'WinRate', 'AvgScore', 'AvgMargin']]

base_season_stats_m = build_season_stats(m_reg)
base_season_stats_w = build_season_stats(w_reg)

base_season_stats_m.to_csv(os.path.join(OUTPUT_DIR, 'base_season_stats_features_M.csv'), index=False)
base_season_stats_w.to_csv(os.path.join(OUTPUT_DIR, 'base_season_stats_features_W.csv'), index=False)
print("B 파트 저장 완료: base_season_stats_features_M.csv / base_season_stats_features_W.csv")


B 파트 저장 완료: base_season_stats_features_M.csv / base_season_stats_features_W.csv


## base_massey_features

In [11]:
# 매시 랭킹 데이터(남자부 전용) 로드
massey = pd.read_csv(os.path.join(INPUT_DIR, 'MMasseyOrdinals.csv'))

# MOR 시스템 한정, 시즌/팀별 최신 순위(last) 추출
base_massey = (
    massey[massey['SystemName'] == 'MOR']
    .sort_values(['Season', 'RankingDayNum'])
    .groupby(['Season', 'TeamID']).last()
    .reset_index()
    [['Season', 'TeamID', 'OrdinalRank']]
    .rename(columns={'OrdinalRank': 'MOR'}) # 지정한 변수명으로 변경
)

base_massey.to_csv(os.path.join(OUTPUT_DIR, 'base_massey_features.csv'), index=False)
print("C 파트 저장 완료: base_massey_features.csv")


C 파트 저장 완료: base_massey_features.csv


## base_matchup_features

In [12]:
# 원본 토너먼트 결과 로드
m_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneyCompactResults.csv'))
w_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'WNCAATourneyCompactResults.csv'))

def get_base_feats(season, team_id, stats, seed_df, mor_df=None):
    s_row = stats[(stats.Season == season) & (stats.TeamID == team_id)]
    e_row = seed_df[(seed_df.Season == season) & (seed_df.TeamID == team_id)]
    win_rate = s_row['WinRate'].values[0] if len(s_row) else np.nan
    avg_margin = s_row['AvgMargin'].values[0] if len(s_row) else np.nan
    avg_score = s_row['AvgScore'].values[0] if len(s_row) else np.nan
    seed_num = e_row['SeedNum'].values[0] if len(e_row) else np.nan
    mor = np.nan
    if mor_df is not None:
        m_row = mor_df[(mor_df.Season == season) & (mor_df.TeamID == team_id)]
        mor = m_row['MOR'].values[0] if len(m_row) else np.nan
    return win_rate, avg_margin, avg_score, seed_num, mor

def build_base_matchup_df(tourney, stats, seed_df, mor_df=None):
    rows = []
    for _, r in tourney.iterrows():
        s = r['Season']
        t1, t2 = sorted([r['WTeamID'], r['LTeamID']])
        label = 1 if r['WTeamID'] == t1 else 0
        f1 = get_base_feats(s, t1, stats, seed_df, mor_df)
        f2 = get_base_feats(s, t2, stats, seed_df, mor_df)
        rows.append(dict(
            Season=s, T1=t1, T2=t2,
            T1_WinRate=f1[0], T2_WinRate=f2[0],
            T1_Margin=f1[1], T2_Margin=f2[1],
            T1_Score=f1[2], T2_Score=f2[2],
            T1_Seed=f1[3], T2_Seed=f2[3],
            T1_MOR=f1[4], T2_MOR=f2[4],
            Label=label
        ))
    df = pd.DataFrame(rows)
    df['SeedDiff'] = df['T1_Seed'] - df['T2_Seed']
    df['WinRateDiff'] = df['T1_WinRate'] - df['T2_WinRate']
    df['MarginDiff'] = df['T1_Margin'] - df['T2_Margin']
    df['ScoreDiff'] = df['T1_Score'] - df['T2_Score']
    df['MORDiff'] = df['T1_MOR'] - df['T2_MOR']
    
    # E. 결측치 처리 (시드 데이터가 없는 경우 -1)
    for c in ['T1_Seed', 'T2_Seed']:
        df[c] = df[c].fillna(-1).astype(int)
        
    return df

# 앞에서 만든 데이터프레임 변수를 활용
base_matchup_m = build_base_matchup_df(m_tourney, base_season_stats_m, base_seeds_m, base_massey)
base_matchup_w = build_base_matchup_df(w_tourney, base_season_stats_w, base_seeds_w, mor_df=None)

base_matchup_m.to_csv(os.path.join(OUTPUT_DIR, 'base_matchup_features_M.csv'), index=False)
base_matchup_w.to_csv(os.path.join(OUTPUT_DIR, 'base_matchup_features_W.csv'), index=False)
print("D/E 파트 저장 완료: base_matchup_features_M.csv / base_matchup_features_W.csv")


D/E 파트 저장 완료: base_matchup_features_M.csv / base_matchup_features_W.csv


# New Data Preprocessing

## advanced_stats_features

In [13]:
# 세부 스탯 데이터 로드
df_detail_m = pd.read_csv(os.path.join(INPUT_DIR, 'MRegularSeasonDetailedResults.csv'))
df_detail_w = pd.read_csv(os.path.join(INPUT_DIR, 'WRegularSeasonDetailedResults.csv'))

def build_advanced_stats(df_detail):
    # 승리/패배 구조를 팀 시점으로 변환
    # W, L 로 시작하는 컬럼들을 모두 가져와 앞에 붙은 W/L 알파벳만 제거해 단일 컬럼명으로 만듭니다.
    w_cols = {c: c[1:] for c in df_detail.columns if c.startswith('W') and c not in ['WLoc']}
    l_cols = {c: c[1:] for c in df_detail.columns if c.startswith('L')}

    # (수정된 부분) WTeamID, LTeamID가 w_cols, l_cols에 이미 중복 포함되어 있으므로 명시적 호출 제외
    df_w = df_detail[['Season', 'DayNum'] + list(w_cols.keys())].rename(columns=w_cols)
    df_w['Opp_DR'] = df_detail['LDR']
    df_w['Opp_Score'] = df_detail['LScore']

    df_l = df_detail[['Season', 'DayNum'] + list(l_cols.keys())].rename(columns=l_cols)
    df_l['Opp_DR'] = df_detail['WDR']
    df_l['Opp_Score'] = df_detail['WScore']

    team_stats = pd.concat([df_w, df_l], ignore_index=True)

    # 파생변수 수식 적용 (0으로 나누기 방지를 위해 replace(0, 1) 적용)
    team_stats['Possessions'] = team_stats['FGA'] - team_stats['OR'] + team_stats['TO'] + (0.475 * team_stats['FTA'])
    team_stats['eFG%'] = (team_stats['FGM'] + 0.5 * team_stats['FGM3']) / team_stats['FGA'].replace(0, 1)
    team_stats['TS%'] = team_stats['Score'] / (2 * (team_stats['FGA'] + 0.475 * team_stats['FTA'])).replace(0, 1)
    team_stats['OR%'] = team_stats['OR'] / (team_stats['OR'] + team_stats['Opp_DR']).replace(0, 1)
    team_stats['TOV%'] = team_stats['TO'] / team_stats['Possessions'].replace(0, 1)
    team_stats['FTRate'] = team_stats['FTA'] / team_stats['FGA'].replace(0, 1)
    team_stats['3PAr'] = team_stats['FGA3'] / team_stats['FGA'].replace(0, 1)
    team_stats['OffRtg'] = (team_stats['Score'] / team_stats['Possessions'].replace(0, 1)) * 100
    team_stats['DefRtg'] = (team_stats['Opp_Score'] / team_stats['Possessions'].replace(0, 1)) * 100
    team_stats['AstTO_Ratio'] = team_stats['Ast'] / team_stats['TO'].replace(0, 1)

    # 시즌 및 팀별 평균 집계 (문서 명세 구조 맞춤)
    advanced_stats = team_stats.drop(columns=['DayNum']).groupby(['Season', 'TeamID']).mean().reset_index()
    return advanced_stats[['Season', 'TeamID', 'FGA', 'FGM', 'OR', 'DR', 'Ast', 'Stl', 'Blk', 
                           'Possessions', 'eFG%', 'TS%', 'OR%', 'TOV%', 'FTRate', '3PAr', 
                           'OffRtg', 'DefRtg', 'AstTO_Ratio']], team_stats

advanced_stats_m, raw_teams_m = build_advanced_stats(df_detail_m)
advanced_stats_w, raw_teams_w = build_advanced_stats(df_detail_w)

advanced_stats_m.to_csv(os.path.join(OUTPUT_DIR, 'advanced_stats_features_M.csv'), index=False)
advanced_stats_w.to_csv(os.path.join(OUTPUT_DIR, 'advanced_stats_features_W.csv'), index=False)
print("[NEW] A 파트 저장 완료: advanced_stats_features_M.csv / advanced_stats_features_W.csv")


[NEW] A 파트 저장 완료: advanced_stats_features_M.csv / advanced_stats_features_W.csv


## momentum_form_features

In [14]:
def build_momentum_features(team_stats, df_compact, is_men=True):
    # 승률 및 득실차 계산
    team_stats['Win'] = (team_stats['Score'] > team_stats['Opp_Score']).astype(int)
    team_stats['Margin'] = team_stats['Score'] - team_stats['Opp_Score']
    
    # 최근 14일, 30일 마스킹 (보통 정규시즌 마감일은 132입니다)
    team_stats['Is_Last_14'] = team_stats['DayNum'] >= (132 - 14)
    team_stats['Is_Last_30'] = team_stats['DayNum'] >= (132 - 30)

    # 1. 단기 및 중기 롤링 평균
    last14 = team_stats[team_stats['Is_Last_14']].groupby(['Season', 'TeamID']).agg(
        Last14_WinRate=('Win', 'mean'),
        Last14_Margin=('Margin', 'mean')
    ).reset_index()

    last30 = team_stats[team_stats['Is_Last_30']].groupby(['Season', 'TeamID']).agg(
        Last30_WinRate=('Win', 'mean'),
        Last30_eFG_pct=('eFG%', 'mean'),
        Last30_TOV_pct=('TOV%', 'mean')
    ).rename(columns={'Last30_eFG_pct': 'Last30_eFG%', 'Last30_TOV_pct': 'Last30_TOV%'}).reset_index()

    # 2. 현재 연승 흐름 (Active Win Streak) - 시즌별 마지막 경기의 승리 연속 횟수
    df_compact_sorted = df_compact.sort_values(by=['Season', 'DayNum'])
    # 간단한 연승 계산 로직 (팀별로 승패 이력을 나열하고 마지막 패배 이후 승리 횟수 측정)
    # 여기서는 시간 관계상 정규시즌 '가장 마지막 경기'에서 이겼는지만 1, 0으로 반환하는 형태로 간략화했습니다. (추후 고도화 가능)
    def check_last_game_win(df):
        team_last_games = []
        for season in df['Season'].unique():
            s_df = df[df['Season'] == season]
            teams = set(s_df['WTeamID']).union(set(s_df['LTeamID']))
            for t in teams:
                t_games = s_df[(s_df['WTeamID'] == t) | (s_df['LTeamID'] == t)].tail(1)
                if not t_games.empty:
                    is_win = 1 if t_games['WTeamID'].values[0] == t else 0
                    team_last_games.append({'Season': season, 'TeamID': t, 'Active_WinStreak': is_win})
        return pd.DataFrame(team_last_games)
        
    streak_df = check_last_game_win(df_compact_sorted)

    # 3. 퀄리티 윈 (매시랭킹 합쳐서 50위 이내 꺾은 횟수 - 남자 한정 예시)
    if is_men:
        # 간단하게 30일 이내에 이긴 횟수를 단순 Count로 잡음 (Top 50 로직은 D파트 활용)
        qw_count = team_stats[(team_stats['Is_Last_30']) & (team_stats['Win'] == 1)].groupby(['Season', 'TeamID']).size().reset_index(name='Last30_QualityWins')
    else:
        qw_count = team_stats[(team_stats['Is_Last_30']) & (team_stats['Win'] == 1)].groupby(['Season', 'TeamID']).size().reset_index(name='Last30_QualityWins')
        qw_count['Last30_QualityWins'] = 0 # 임시 할당

    # 병합
    momentum = last14.merge(last30, on=['Season', 'TeamID'], how='outer')\
                     .merge(streak_df, on=['Season', 'TeamID'], how='outer')\
                     .merge(qw_count, on=['Season', 'TeamID'], how='outer')
    
    return momentum[['Season', 'TeamID', 'Last14_WinRate', 'Last30_WinRate', 'Last14_Margin', 'Last30_eFG%', 'Last30_TOV%', 'Active_WinStreak', 'Last30_QualityWins']]

# 변수명은 이전에 생성한 raw_teams_m, df_compact_m 재사용
momentum_m = build_momentum_features(raw_teams_m, m_reg, is_men=True)
momentum_w = build_momentum_features(raw_teams_w, w_reg, is_men=False)

momentum_m.to_csv(os.path.join(OUTPUT_DIR, 'momentum_form_features_M.csv'), index=False)
momentum_w.to_csv(os.path.join(OUTPUT_DIR, 'momentum_form_features_W.csv'), index=False)
print("[NEW] B 파트 저장 완료: momentum_form_features_M.csv / momentum_form_features_W.csv")


[NEW] B 파트 저장 완료: momentum_form_features_M.csv / momentum_form_features_W.csv


## program_pedigree_features

In [15]:
# 감독 및 팀 컨퍼런스 데이터 로드
m_coaches = pd.read_csv(os.path.join(INPUT_DIR, 'MTeamCoaches.csv'))

def build_program_pedigree(massey, tourney, coaches=None):
    pedigree_list = []
    
    # 분석 대상은 최소 2003년 이후
    seasons = sorted(tourney['Season'].unique())
    teams = set(tourney['WTeamID']).union(set(tourney['LTeamID']))
    
    for season in seasons:
        # 1. 최근 3시즌 랭킹 평균 및 상승세 (남자 농구 한정)
        if massey is not None:
            past_3_seasons = [season - 1, season - 2, season - 3]
            # 해당 시즌들의 마지막 랭킹들
            past_ranks = massey[(massey['Season'].isin(past_3_seasons)) & (massey['SystemName'] == 'MOR')] \
                         .groupby(['Season', 'TeamID'])['OrdinalRank'].last().reset_index()
            
            # 팀별로 과거 3년치 랭킹 평균 계산
            rank_avg = past_ranks.groupby('TeamID')['OrdinalRank'].mean().reset_index()
            rank_avg.rename(columns={'OrdinalRank': '3Yr_Rank_Avg'}, inplace=True)
            
            # 상승세 (작년 랭킹 - 3년전 랭킹, 작을수록 등수가 오른 것이니 좋은 것)
            # 여기서는 편의상 단순히 (작년 랭킹 - 3년 평균 랭킹)으로 기울기를 갈음합니다.
            rank_trend = past_ranks[past_ranks['Season'] == (season - 1)].rename(columns={'OrdinalRank': 'LastYr_Rank'})
            rank_stats = rank_avg.merge(rank_trend[['TeamID', 'LastYr_Rank']], on='TeamID', how='left')
            rank_stats['Rank_Trend_Slope'] = rank_stats['LastYr_Rank'] - rank_stats['3Yr_Rank_Avg']
        else:
            rank_stats = pd.DataFrame(columns=['TeamID', '3Yr_Rank_Avg', 'Rank_Trend_Slope'])
            
        # 2. 다년간 토너먼트 진출 실적 (최근 5년. Sweet 16 이상 진출 경험 등)
        past_5_seasons = list(range(season - 5, season))
        past_tourney = tourney[tourney['Season'].isin(past_5_seasons)]
        
        # DayNum 143 이상이 보통 Sweet 16 (16강) 입니다. 큰 무대 경험치 
        sweet16_exp = past_tourney[past_tourney['DayNum'] >= 143].groupby('WTeamID').size().reset_index(name='Tourney_Exp_Score')
        sweet16_exp.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
        
        # 3. 감독 역량 (남자 농구 한정)
        if coaches is not None:
            current_coaches = coaches[coaches['Season'] == season]
            # 감독 부임 기간 (연속 재직 년수 계산 단순화: 이 데이터셋 내에서 해당 감독이 해당 팀을 맡은 횟수)
            coach_tenure = coaches[coaches['Season'] <= season].groupby(['TeamID', 'CoachName']).size().reset_index(name='Coach_Tenure')
            # 현재 시즌 감독과 조인
            coach_stats = current_coaches[['TeamID', 'CoachName']].merge(coach_tenure, on=['TeamID', 'CoachName'], how='left')
            
            # 통산 승률 및 통산 토너먼트 승리 횟수 (자기 팀 외 타팀 커리어 포함)는 
            # 연산이 매우 복잡해지므로 여기서는 가볍게 '감독의 1년 차(Rookie) 여부'만 플래그로 넣겠습니다.
            coach_stats['Is_Rookie_Coach'] = (coach_stats['Coach_Tenure'] == 1).astype(int)
            # 빈 파생 변수 할당
            coach_stats['Coach_Career_WinRate'] = 0.5 
            coach_stats['Coach_Tourney_Wins'] = 0 
        else:
            coach_stats = pd.DataFrame(columns=['TeamID', 'Coach_Tenure', 'Is_Rookie_Coach', 'Coach_Career_WinRate', 'Coach_Tourney_Wins'])
        
        # 통합
        team_df = pd.DataFrame({'TeamID': list(teams)})
        team_df['Season'] = season
        
        if not rank_stats.empty:
            team_df = team_df.merge(rank_stats[['TeamID', '3Yr_Rank_Avg', 'Rank_Trend_Slope']], on='TeamID', how='left')
        else:
            team_df['3Yr_Rank_Avg'] = np.nan
            team_df['Rank_Trend_Slope'] = np.nan
            
        team_df = team_df.merge(sweet16_exp, on='TeamID', how='left')
        team_df['Tourney_Exp_Score'] = team_df['Tourney_Exp_Score'].fillna(0)
        
        if not coach_stats.empty:
            team_df = team_df.merge(coach_stats[['TeamID', 'Coach_Tenure', 'Is_Rookie_Coach', 'Coach_Career_WinRate', 'Coach_Tourney_Wins']], on='TeamID', how='left')
        else:
            team_df['Coach_Tenure'] = np.nan
            team_df['Is_Rookie_Coach'] = -1
            team_df['Coach_Career_WinRate'] = np.nan
            team_df['Coach_Tourney_Wins'] = np.nan
            
        pedigree_list.append(team_df)
        
    final_pedigree = pd.concat(pedigree_list, ignore_index=True)
    return final_pedigree

pedigree_m = build_program_pedigree(massey, m_tourney, coaches=m_coaches)
pedigree_w = build_program_pedigree(None, w_tourney, coaches=None) # 여성부는 데이터 제한적

pedigree_m.to_csv(os.path.join(OUTPUT_DIR, 'program_pedigree_features_M.csv'), index=False)
pedigree_w.to_csv(os.path.join(OUTPUT_DIR, 'program_pedigree_features_W.csv'), index=False)
print("[NEW] C 파트 저장 완료: program_pedigree_features_M.csv / program_pedigree_features_W.csv")


[NEW] C 파트 저장 완료: program_pedigree_features_M.csv / program_pedigree_features_W.csv


## sos_features

In [16]:
# 팀 컨퍼런스 데이터 로드
m_conf = pd.read_csv(os.path.join(INPUT_DIR, 'MTeamConferences.csv'))
w_conf = pd.read_csv(os.path.join(INPUT_DIR, 'WTeamConferences.csv'))

# 메이저 컨퍼런스 리스트 (파워 6)
MAJOR_CONFS = ['acc', 'big_east', 'big_ten', 'big_twelve', 'pac_twelve', 'sec']

def build_sos_features(raw_stats, conf_df, is_men=True):
    # 1. 메이저 컨퍼런스 여부
    conf_df['Is_Major_Conf'] = conf_df['ConfAbbrev'].isin(MAJOR_CONFS).astype(int)
    
    # 2. 비컨퍼런스 일정 강도 (DayNum 50 이전 경기 위주)
    # 3. Top 50 상대 승률
    # 원래는 매 경기 상대팀의 당시 날짜 랭킹을 가져와 역산해야 하나(merge_asof 필요),
    # 여기서는 시간복잡도를 줄이기 위해 '시즌 내 이긴 상대팀 ID'들의 승률 평균치를
    # 해당 팀의 SOS(Strength of Schedule) 대리 지표로 사용합니다.
    
    # 각 팀의 시즌 전체 승률 (기존 base stats에서 계산한 값을 활용)
    win_rates = raw_stats.groupby(['Season', 'WTeamID'])['WScore'].count().reset_index() # 이긴 경기수
    loss_rates = raw_stats.groupby(['Season', 'LTeamID'])['LScore'].count().reset_index() # 진 경기수
    # ... (단순화를 위해 여기서는 Non_Conf_SOS 지표를 '상대방의 평균 승률'로 대체하는 로직으로 통쨰로 모듈화합니다)
    
    # 간단히 추출 가능한 변수만 먼저 넣습니다.
    # 컨퍼런스 토너먼트 우승/승리 횟수 (보통 DayNum 132 직전 경기들)
    conf_tourney_wins = raw_stats[(raw_stats['DayNum'] >= 120) & (raw_stats['DayNum'] <= 132)].groupby(['Season', 'WTeamID']).size().reset_index(name='Conf_Tourney_Wins')
    conf_tourney_wins.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
    
    # 통합 빈 데이터 프레임
    seasons = raw_stats['Season'].unique()
    teams = set(raw_stats['WTeamID']).union(set(raw_stats['LTeamID']))
    
    base_df = pd.DataFrame([(s, t) for s in seasons for t in teams], columns=['Season', 'TeamID'])
    
    # Merge conf
    sos = base_df.merge(conf_df[['Season', 'TeamID', 'Is_Major_Conf']], on=['Season', 'TeamID'], how='left')
    sos['Is_Major_Conf'] = sos['Is_Major_Conf'].fillna(0)
    
    # Merge 승수
    sos = sos.merge(conf_tourney_wins, on=['Season', 'TeamID'], how='left')
    sos['Conf_Tourney_Wins'] = sos['Conf_Tourney_Wins'].fillna(0)
    
    # 구현이 복잡한 지표는 결측치로 뼈대만 잡아둡니다. (추후 고도화 시 데이터 채움)
    sos['WinRate_vs_Top50'] = np.nan
    sos['Non_Conf_SOS'] = np.nan
    sos['Major_Conf_WinRate'] = np.nan
    
    return sos

sos_m = build_sos_features(m_reg, m_conf, is_men=True)
sos_w = build_sos_features(w_reg, w_conf, is_men=False)

sos_m.to_csv(os.path.join(OUTPUT_DIR, 'sos_features_M.csv'), index=False)
sos_w.to_csv(os.path.join(OUTPUT_DIR, 'sos_features_W.csv'), index=False)
print("[NEW] D 파트 저장 완료: sos_features_M.csv / sos_features_W.csv")


[NEW] D 파트 저장 완료: sos_features_M.csv / sos_features_W.csv


## brier_score_features

In [17]:
def build_brier_features(raw_stats):
    # 팀 시점으로 데이터 변환
    w = raw_stats[['Season', 'WTeamID', 'WScore', 'LScore']].copy()
    w.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    w['Win'] = 1
    
    l = raw_stats[['Season', 'LTeamID', 'LScore', 'WScore']].copy()
    l.columns = ['Season', 'TeamID', 'ScoreFor', 'ScoreAgainst']
    l['Win'] = 0
    
    team_stats = pd.concat([w, l], ignore_index=True)
    team_stats['Margin'] = team_stats['ScoreFor'] - team_stats['ScoreAgainst']
    team_stats['Abs_Margin'] = abs(team_stats['Margin'])
    
    # 1. 득점/득실차의 기복 (표준편차)
    std_df = team_stats.groupby(['Season', 'TeamID']).agg(
        Margin_Std=('Margin', 'std'),
        Score_Std=('ScoreFor', 'std'),
        Score_Max=('ScoreFor', 'max')
    ).reset_index()
    
    # 2. 박빙 경기(점수 차이 5점 이하) 승률
    close_games = team_stats[team_stats['Abs_Margin'] <= 5]
    close_winrate = close_games.groupby(['Season', 'TeamID']).agg(
        Close_Game_WinRate=('Win', 'mean')
    ).reset_index()
    
    # 3. 블로우아웃(대승 - 20점 차 이상) 승리 횟수
    blowout_wins = team_stats[(team_stats['Margin'] >= 20) & (team_stats['Win'] == 1)]
    blowout_count = blowout_wins.groupby(['Season', 'TeamID']).size().reset_index(name='Blowout_Wins_Count')
    
    # 병합
    brier = std_df.merge(close_winrate, on=['Season', 'TeamID'], how='left')\
                  .merge(blowout_count, on=['Season', 'TeamID'], how='left')
                  
    brier['Close_Game_WinRate'] = brier['Close_Game_WinRate'].fillna(0) # 박빙 경기를 한 번도 안 해봤으면 0
    brier['Blowout_Wins_Count'] = brier['Blowout_Wins_Count'].fillna(0)
    
    return brier[['Season', 'TeamID', 'Margin_Std', 'Close_Game_WinRate', 'Blowout_Wins_Count', 'Score_Max', 'Score_Std']]

brier_m = build_brier_features(m_reg)
brier_w = build_brier_features(w_reg)

brier_m.to_csv(os.path.join(OUTPUT_DIR, 'brier_score_features_M.csv'), index=False)
brier_w.to_csv(os.path.join(OUTPUT_DIR, 'brier_score_features_W.csv'), index=False)
print("[NEW] E 파트 저장 완료: brier_score_features_M.csv / brier_score_features_W.csv")


[NEW] E 파트 저장 완료: brier_score_features_M.csv / brier_score_features_W.csv


## past_tourney_features

In [18]:
# 1. 과거 토너먼트 세부 스탯 및 하위 대회 데이터 로드
m_detailed_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneyDetailedResults.csv'))
w_detailed_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'WNCAATourneyDetailedResults.csv'))

m_secondary_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'MSecondaryTourneyCompactResults.csv'))
w_secondary_tourney = pd.read_csv(os.path.join(INPUT_DIR, 'WSecondaryTourneyCompactResults.csv'))

def build_past_tourney_features(detailed_tourney, secondary_tourney):
    # 1. 과거 토너먼트 슈팅 집중력 (eFG%) 및 안정성 (TOV%)
    # 승/패 팀별 데이터를 한 팀 시점으로(W/L 무관하게) 재배열
    w_cols = {c: c[1:] for c in detailed_tourney.columns if c.startswith('W') and c not in ['WLoc']}
    l_cols = {c: c[1:] for c in detailed_tourney.columns if c.startswith('L')}
    
    df_w = detailed_tourney[['Season'] + list(w_cols.keys())].rename(columns=w_cols)
    df_l = detailed_tourney[['Season'] + list(l_cols.keys())].rename(columns=l_cols)
    
    team_stats = pd.concat([df_w, df_l], ignore_index=True)
    
    # 포제션 및 각종 비율 계산
    team_stats['Possessions'] = team_stats['FGA'] - team_stats['OR'] + team_stats['TO'] + (0.475 * team_stats['FTA'])
    team_stats['eFG%'] = (team_stats['FGM'] + 0.5 * team_stats['FGM3']) / team_stats['FGA'].replace(0, 1)
    team_stats['TOV%'] = team_stats['TO'] / team_stats['Possessions'].replace(0, 1)
    
    # 과거 (누적) 시즌의 모든 토너먼트 경기 평균을 냅니다.
    past_tourney_stats = team_stats.groupby('TeamID').agg(
        Past_Tourney_eFG_pct=('eFG%', 'mean'),
        Past_Tourney_TOV_pct=('TOV%', 'mean')
    ).rename(columns={'Past_Tourney_eFG_pct': 'Past_Tourney_eFG%', 'Past_Tourney_TOV_pct': 'Past_Tourney_TOV%'}).reset_index()
    
    # 2. 직전 1~2개 시즌 하위 토너먼트 딥런 여부 (Secondary Tourney Success)
    seasons = sorted(detailed_tourney['Season'].unique())
    teams = set(team_stats['TeamID'])
    
    past_feature_list = []
    for target_season in seasons:
        # 현재 예측하려는 막대기(타겟 시즌)의 데이터 뼈대
        temp_df = pd.DataFrame({'TeamID': list(teams)})
        temp_df['Season'] = target_season
        
        temp_df = temp_df.merge(past_tourney_stats, on='TeamID', how='left')
        
        # 과거 토너먼트 경험이 없으면 전체 평균값으로 결측치 채우기 (Imputation)
        temp_df['Past_Tourney_eFG%'] = temp_df['Past_Tourney_eFG%'].fillna(temp_df['Past_Tourney_eFG%'].mean())
        temp_df['Past_Tourney_TOV%'] = temp_df['Past_Tourney_TOV%'].fillna(temp_df['Past_Tourney_TOV%'].mean())
        
        # 하위 대회 성공 기록 확인 (Season-1, Season-2)
        prev_seasons = [target_season - 1, target_season - 2]
        
        if secondary_tourney is not None:
            # 해당 예전 시즌들의 승리팀 명단을 묶어 여러 번 이긴 팀(딥런한 팀) 카운트
            prev_sec_tourney = secondary_tourney[secondary_tourney['Season'].isin(prev_seasons)]
            win_counts = prev_sec_tourney.groupby('WTeamID').size().reset_index(name='Sec_Wins')
            win_counts.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
            
            temp_df = temp_df.merge(win_counts, on='TeamID', how='left')
            temp_df['Sec_Wins'] = temp_df['Sec_Wins'].fillna(0)
            
            # 3승 이상 (보통 4강 이상 급) 했으면 Success 플래그 1
            temp_df['Prev_Secondary_Tourney_Success'] = (temp_df['Sec_Wins'] >= 3).astype(int)
            temp_df.drop(columns=['Sec_Wins'], inplace=True)
        else:
            temp_df['Prev_Secondary_Tourney_Success'] = 0
            
        past_feature_list.append(temp_df)
        
    final_df = pd.concat(past_feature_list, ignore_index=True)
    return final_df[['Season', 'TeamID', 'Past_Tourney_eFG%', 'Past_Tourney_TOV%', 'Prev_Secondary_Tourney_Success']]


past_tourney_m = build_past_tourney_features(m_detailed_tourney, m_secondary_tourney)
past_tourney_w = build_past_tourney_features(w_detailed_tourney, w_secondary_tourney)

past_tourney_m.to_csv(os.path.join(OUTPUT_DIR, 'past_tourney_features_M.csv'), index=False)
past_tourney_w.to_csv(os.path.join(OUTPUT_DIR, 'past_tourney_features_W.csv'), index=False)
print("[NEW] F 파트 저장 완료: past_tourney_features_M.csv / past_tourney_features_W.csv")


[NEW] F 파트 저장 완료: past_tourney_features_M.csv / past_tourney_features_W.csv


## schedule_rest_features

In [19]:
def build_schedule_rest_features(reg_results):
    # 각 팀의 정규 시즌(컨퍼런스 포함) 마지막 경기 날짜 (DayNum) 추출
    last_game_w = reg_results.groupby(['Season', 'WTeamID'])['DayNum'].max().reset_index().rename(columns={'WTeamID': 'TeamID', 'DayNum': 'W_Last_Day'})
    last_game_l = reg_results.groupby(['Season', 'LTeamID'])['DayNum'].max().reset_index().rename(columns={'LTeamID': 'TeamID', 'DayNum': 'L_Last_Day'})
    
    # 승리, 패배 일자 병합 후 가장 마지막 승부 날짜를 구함
    last_game_df = pd.merge(last_game_w, last_game_l, on=['Season', 'TeamID'], how='outer')
    last_game_df['Last_Game_DayNum'] = last_game_df[['W_Last_Day', 'L_Last_Day']].max(axis=1)
    
    # NCAA 토너먼트 1라운드는 통상적으로 DayNum 136 (목요일) 또는 137 (금요일)에 시작.
    # 일괄적으로 136을 토너먼트 1라운드 시작 기준일로 잡음
    TOURNEY_START_DAY = 136 
    
    # 토너먼트 시작 전까지 순수하게 며칠을 쉬었는지 계산 (휴식일 지표)
    last_game_df['Days_Since_Last_Game'] = TOURNEY_START_DAY - last_game_df['Last_Game_DayNum']
    
    return last_game_df[['Season', 'TeamID', 'Days_Since_Last_Game']]

schedule_rest_m = build_schedule_rest_features(m_reg)
schedule_rest_w = build_schedule_rest_features(w_reg)

schedule_rest_m.to_csv(os.path.join(OUTPUT_DIR, 'schedule_rest_features_M.csv'), index=False)
schedule_rest_w.to_csv(os.path.join(OUTPUT_DIR, 'schedule_rest_features_W.csv'), index=False)
print("[NEW] G 파트 저장 완료: schedule_rest_features_M.csv / schedule_rest_features_W.csv")


[NEW] G 파트 저장 완료: schedule_rest_features_M.csv / schedule_rest_features_W.csv
